# A Regression Tree

**Notebook 3 of 5.** The same algorithm predicts a number with two changes: the leaf returns
a **mean** instead of a majority, and impurity is **variance** instead of Gini.

The task: **predict the compressive strength of concrete** from its recipe — a strongly
non-linear problem, which is where trees earn their place.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

## 1. The data

1,030 laboratory mixes: seven ingredients in kilograms per cubic meter, the sample's age in
days, and `compressive_strength` in megapascals — the pressure it withstood before breaking.

The people who collected it call strength *"a highly non-linear function of age and
ingredients"*.

In [ ]:
df = pd.read_csv("data/concrete.csv")
print(df.shape)
df.head()

In [ ]:
df.describe().T.round(2)

No missing values, all numeric, target from 2.33 to 82.60 MPa around a mean of 35.82. Slag,
fly ash and superplasticizer are zero in many mixes: not every recipe uses them.

In [ ]:
# the eight ingredients and the age; strength is what we predict
X = df.drop(columns="compressive_strength")
y = df["compressive_strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"{len(X_train)} training mixes, {len(X_test)} test mixes")

> **Note.** No `stratify` here: it balances *classes*, and a continuous target has none.

## 2. What changes inside the tree

Two things, and nothing else.

**The leaf predicts the mean** of the training mixes that reach it — so a regression tree can
only predict values it has averaged from training data.

**Impurity becomes variance.** A node is pure when all of its rows share one target value,
so the measure is the mean squared error around the node's own mean. For the $n$ rows in a
node, each with target value $y_i$, and $\bar{y}_{node}$ their mean:

$$\text{MSE}(node) = \frac{1}{n}\sum_{i \in node} (y_i - \bar{y}_{node})^2$$

The two sides of a split combine by the same weighted sum as before, and the largest
reduction wins. That is `criterion="squared_error"`, the default; `"absolute_error"` and
`"poisson"` exist for targets where squared error is the wrong loss.

## 3. One feature, so the shape is visible

Cement alone, two questions deep.

In [ ]:
# one feature and two questions deep, so the staircase has only four steps
cement_tree = DecisionTreeRegressor(max_depth=2, random_state=42)
cement_tree.fit(X_train[["cement"]], y_train)

fig, ax = plt.subplots(figsize=(12, 6))
plot_tree(cement_tree, feature_names=["cement"], filled=True, ax=ax)
plt.show()

The boxes hold what the classification tree's did, with two changes:

- **`squared_error`** stands where `gini` stood — the impurity of a node is now the variance
  of its mixes around their own mean;
- **`value`** is a single number, the mean strength of the rows that landed there, and it is
  what the leaf predicts.

Read the leaves left to right and you have the four steps the next plot draws: **28.04**,
**37.36**, **58.18** and **46.91** MPa.

In [ ]:
# a prediction every kilogram across the training range, to draw the steps
grid = pd.DataFrame({"cement": np.arange(100, 560, 1)})

plt.figure(figsize=(8, 5))
plt.scatter(X_train["cement"], y_train, s=12, alpha=0.4, color="#252629", label="training mixes")
plt.plot(grid["cement"], cement_tree.predict(grid), color="#FF4A11", linewidth=2.5, label="tree, depth 2")
plt.xlabel("cement (kg/m³)"); plt.ylabel("compressive strength (MPa)")
plt.legend(); plt.title("What a regression tree predicts")
plt.show()

**A staircase, not a curve.** Four leaves, four flat steps, jumping at the thresholds the
tree chose: 257.70, 352.50, 374.15. Inside a step every mix gets the same answer.

That is *piecewise constant*, the defining shape of a tree regressor. More depth means
narrower steps, never a slope.

## 4. All eight features

Two reference points first: always predicting the mean, and a linear regression.

In [ ]:
def report(name, model):
    """Print train and test error for a fitted regressor.

    Args:
        name: label for the printed row.
        model: a fitted scikit-learn regressor.
    """

    # both halves, since the gap between them is what the report is for
    train_pred, test_pred = model.predict(X_train), model.predict(X_test)
    print(
        f"{name:22s} train MAE {mean_absolute_error(y_train, train_pred):5.2f}"
        f"   test MAE {mean_absolute_error(y_test, test_pred):5.2f}"
        f"   test RMSE {root_mean_squared_error(y_test, test_pred):5.2f}"
        f"   test R² {r2_score(y_test, test_pred):6.3f}"
    )


report("mean baseline", DummyRegressor().fit(X_train, y_train))
report("linear regression", LinearRegression().fit(X_train, y_train))

The three numbers:

- **MAE**, the mean absolute error, is the average miss in MPa, readable in the units of the
  problem;
- **RMSE** squares the errors before averaging, so a few large misses hurt it more than MAE;
- **R²** is the share of the variance explained: 0 is the mean baseline, 1 is perfect, and
  negative is worse than guessing the mean.

Linear regression explains 63% and misses by 7.75 MPa on average.

> **Exercise:**
>
> Fit a `DecisionTreeRegressor` with no limits, print its depth and leaf count, and pass it
> to `report`. How does it compare with linear regression, and what does its *training* error
> tell you?

<details><summary>
Click here for a hint...
</summary>

`DecisionTreeRegressor(random_state=42).fit(X_train, y_train)`, then `tree_.max_depth`,
`get_n_leaves()` and `report(...)`.

</details>

In [ ]:
# your code: fit an unlimited regression tree, print its size, and report its errors

<details><summary>
Click here to compare your answer...
</summary>

**Depth 18, 791 leaves** for 824 training mixes — almost one leaf each. Training MAE **0.06
MPa** is not a pattern, it is a lookup table.

And yet test MAE is **4.48** against linear regression's 7.75, R² **0.803** against 0.628.
Strength really is non-linear in the recipe, and a linear model cannot bend.

</details>

## 5. Does limiting the depth help here?

In [ ]:
depths = range(1, 16)
train_rmse, test_rmse = [], []

# one tree per depth, both halves scored, as on the census data
for depth in depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_rmse.append(root_mean_squared_error(y_train, model.predict(X_train)))
    test_rmse.append(root_mean_squared_error(y_test, model.predict(X_test)))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_rmse, marker="o", label="train")
plt.plot(depths, test_rmse, marker="o", label="test")
plt.xlabel("max_depth"); plt.ylabel("RMSE (MPa)")
plt.xticks(list(depths))
plt.legend(); plt.title("Error against tree depth")
plt.show()

best = depths[int(np.argmin(test_rmse))]
print(f"best depth: {best}, test RMSE {min(test_rmse):.2f}")

The training curve falls to almost nothing, as on the census data. The test curve barely
punishes it: best is **depth 13 at RMSE 6.7576**, depth 9 a thousandth behind, the unlimited
tree **7.13**. Past depth 8 the curve is flat and bumpy — those gaps are noise.

**How much overfitting costs depends on the data.** The same gap cost four points on the
census records and almost nothing here, because the recipe genuinely determines the strength
and there is little noise to memorize. Draw the curve rather than assuming.

## 6. The one thing a tree cannot do

A leaf holds the mean of the rows that reached it. Beyond the training range there are no
rows and no new mean, so the tree returns its last one, forever.

In [ ]:
# 540 is the edge of the training data; everything past it is extrapolation
far_grid = pd.DataFrame({"cement": [300, 540, 700, 1000]})

linear_cement = LinearRegression().fit(X_train[["cement"]], y_train)

comparison = pd.DataFrame({
    "cement": far_grid["cement"],
    "tree": cement_tree.predict(far_grid).round(2),
    "linear": linear_cement.predict(far_grid).round(2),
})
print(f"most cement in any training mix: {X_train['cement'].max()} kg/m³")
comparison

At 540 kg/m³, the edge of the data, both answer. Past it the linear model keeps climbing —
68 MPa, then 92 — while **the tree repeats 46.91 forever**, the mean of its last leaf.

Neither is trustworthy out there. The difference is that the linear model extrapolates a
trend and the tree does not extrapolate at all.

> **Exercise:**
>
> Plot both models across `cement` from 100 to 1000 on the same axes, training mixes
> underneath, so the flat tail and the straight line sit side by side.

<details><summary>
Click here for a hint...
</summary>

One `DataFrame({"cement": np.arange(100, 1001, 1)})`, `.predict` with each model, and an
`axvline` at `X_train["cement"].max()`.

</details>

In [ ]:
# your code: plot the tree and the linear model from cement 100 to 1000

<details><summary>
Click here to compare your answer...
</summary>

Past the dashed line the tree is a horizontal ray; the linear model climbs to 92 MPa, a
strength no mix ever reached.

This matters beyond concrete. Any target that **trends** — sales growing year on year, a
sensor drifting — eventually leaves the range the tree was trained on, and there it flattens.
Ask whether what you are predicting will stay inside that range.

</details>

## Summary

- Same algorithm: leaves predict the **mean**, impurity is **variance** rather than Gini.
- Predictions are **piecewise constant**. More depth means more steps, never a slope.
- On concrete an unlimited tree reached **MAE 4.48, R² 0.803**, well ahead of linear
  regression's 7.75 and 0.628 — the relationship is genuinely non-linear.
- It memorized the training set to **MAE 0.06** and barely paid for it. Overfitting costs
  different amounts on different data; draw the curve.
- **A tree cannot extrapolate**, and that is the one failure you cannot tune away.

Next, in `04_reading_a_fitted_tree.ipynb`: which features it leaned on, and why the obvious
way to measure that misleads.